# Sign Language Model — Webcam Test (Google Colab)

Tests your trained ISL sign recognition model using your **laptop/webcam** through the browser (Colab has no direct camera access, so we grab frames via JavaScript).

Model: Keras LSTM, input shape `(60, 258)` — MediaPipe Holistic keypoints (pose + both hands), classes:
`COLD, DAYS, EAT, FEVER, HEADACHE, MEDICINE, NEUTRAL, NOT, THREE, THROATPAIN, TWO`

**Before running:** upload your trained `.h5`/`.keras` model file when prompted in Step 2.


## Step 1 — Install dependencies

In [ ]:
!pip install -q mediapipe opencv-python-headless


## Step 2 — Upload your trained model

In [ ]:
from google.colab import files
import tensorflow as tf

print("Upload your trained model file (.h5 or .keras)")
uploaded = files.upload()
MODEL_PATH = list(uploaded.keys())[0]

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()


## Step 3 — Config (must match training exactly)

Edit `LABEL_MAP` if your class order differs from training. Order matters — it must match
the order used when you one-hot encoded `y` during training.

In [ ]:
SEQUENCE_LENGTH = 60
NEUTRAL_LABEL = "NEUTRAL"
CONFIDENCE_THRESHOLD = 0.80  # match whatever you used in main.py

LABEL_MAP = ['COLD', 'DAYS', 'EAT', 'FEVER', 'HEADACHE', 'MEDICINE',
             'NEUTRAL', 'NOT', 'THREE', 'THROATPAIN', 'TWO']

NO_HAND_FRAMES_TO_END_SIGN = 6   # ~6 frames of no hands = sign finished
MIN_SIGN_FRAMES = 10             # ignore tiny accidental blips


## Step 4 — Keypoint extraction, resampling, normalization

`extract_keypoints` below is the standard MediaPipe Holistic layout (pose 33×4=132 +
left hand 21×3=63 + right hand 21×3=63 = 258), matching your feature count.

⚠️ **`normalize_sequence` is a placeholder.** Your exported chat referenced this function
by name but the exact body wasn't in the export. If your training pipeline normalized
keypoints (e.g. subtracted a wrist reference point, divided by a scale factor), paste
that exact function here — otherwise predictions will not match your local results.

In [ ]:
import mediapipe as mp
import numpy as np

mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def extract_keypoints(results):
    pose = np.array([[r.x, r.y, r.z, r.visibility] for r in results.pose_landmarks.landmark]).flatten() \
        if results.pose_landmarks else np.zeros(33 * 4)
    lh = np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten() \
        if results.left_hand_landmarks else np.zeros(21 * 3)
    rh = np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten() \
        if results.right_hand_landmarks else np.zeros(21 * 3)
    return np.concatenate([pose, lh, rh])


def resample_to_length(frames, target_length):
    sequence = np.array(frames)
    original_len = sequence.shape[0]
    old_idx = np.linspace(0, original_len - 1, original_len)
    new_idx = np.linspace(0, original_len - 1, target_length)
    resampled = np.zeros((target_length, sequence.shape[1]))
    for feat in range(sequence.shape[1]):
        resampled[:, feat] = np.interp(new_idx, old_idx, sequence[:, feat])
    return resampled


def normalize_sequence(sequence):
    # TODO: replace with your ACTUAL training-time normalization if you used one.
    # Left as identity for now.
    return sequence


## Step 5 — Browser webcam capture (JavaScript)

`take_photo_js()` grabs a single frame from your webcam through the browser and returns
it as a numpy BGR image, callable repeatedly from Python — same idea as your FastAPI
`/predict-frame` endpoint, just sourcing frames from the browser instead of a local camera.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2

def _init_camera():
    js = Javascript('''
        var video;
        var stream;
        async function initCamera() {
            video = document.createElement('video');
            video.style.display = 'block';
            stream = await navigator.mediaDevices.getUserMedia({video: true});
            document.body.appendChild(video);
            video.srcObject = stream;
            await video.play();
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        }
        async function takePhoto() {
            var canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            return canvas.toDataURL('image/jpeg', 0.9);
        }
        function stopCamera() {
            if (stream) { stream.getTracks().forEach(t => t.stop()); }
            if (video) { video.remove(); }
        }
    ''')
    display(js)
    eval_js('initCamera()')

def take_photo_js():
    data_url = eval_js('takePhoto()')
    header, encoded = data_url.split(',', 1)
    jpg_bytes = b64decode(encoded)
    np_arr = np.frombuffer(jpg_bytes, np.uint8)
    frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
    return frame

def stop_camera_js():
    eval_js('stopCamera()')


## Step 6 — Run the live test

Starts your webcam, then repeatedly grabs frames and runs them through the SAME
segment-based logic as your local `main.py`: buffer while hands are visible, and once
hands disappear for a few frames, resample the segment to 60 frames and predict once.

**How to sign:** lower your hand out of frame briefly between each sign — that triggers
"sign ended" detection, same as your local setup. Run the cell, then interrupt it
(Runtime → Interrupt execution, or the ■ stop button) when you're done testing.

In [ ]:
import time

_init_camera()
time.sleep(2)  # let the camera warm up

buffer_frames = []
is_signing = False
no_hand_counter = 0
committed_sequence = []

print("Camera running. Sign, then drop your hand out of frame to end each sign.")
print("Interrupt this cell (stop button) to end the session.\n")

try:
    while True:
        frame = take_photo_js()
        if frame is None:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
        hands_present = bool(results.left_hand_landmarks or results.right_hand_landmarks)
        keypoints = extract_keypoints(results)

        if hands_present:
            is_signing = True
            no_hand_counter = 0
            buffer_frames.append(keypoints)
        elif is_signing:
            no_hand_counter += 1
            buffer_frames.append(keypoints)

            if no_hand_counter >= NO_HAND_FRAMES_TO_END_SIGN:
                if len(buffer_frames) >= MIN_SIGN_FRAMES:
                    sequence = resample_to_length(buffer_frames, SEQUENCE_LENGTH)
                    sequence = normalize_sequence(sequence)
                    input_data = np.expand_dims(sequence, axis=0)

                    prediction = model.predict(input_data, verbose=0)[0]
                    pred_idx = int(np.argmax(prediction))
                    confidence = float(prediction[pred_idx])
                    pred_label = LABEL_MAP[pred_idx]

                    print(f"Segment finished ({len(buffer_frames)} frames) -> {pred_label} ({confidence:.2%})")

                    if confidence >= CONFIDENCE_THRESHOLD and pred_label != NEUTRAL_LABEL:
                        if not committed_sequence or committed_sequence[-1] != pred_label:
                            committed_sequence.append(pred_label)
                            print(f"  Committed sequence: {committed_sequence}")

                buffer_frames = []
                is_signing = False
                no_hand_counter = 0

        time.sleep(0.05)  # ~20 fps request rate; adjust if the browser lags

except KeyboardInterrupt:
    pass
finally:
    stop_camera_js()
    print("\nCamera stopped. Final glosses:", committed_sequence)


## Notes

- **Speed:** each loop iteration does a JS round-trip + MediaPipe inference, so Colab's
  webcam feed will be slower than your local FastAPI setup (maybe 5-15 fps depending on
  connection). This is fine for testing accuracy, not for production latency testing.
- **`normalize_sequence`:** fix the placeholder in Step 4 before trusting predictions.
- **Confidence threshold:** currently `0.80` — adjust in Step 3 to match `main.py`.
- If Colab prompts for camera/microphone permission in the browser, allow it — this only
  happens once per session.
